# 01 · Getting started

The basics on one table at a time: looking around, filtering, sorting, and adding things up.

**How to use this notebook.** Run a cell with **Shift+Enter**. The first time, VS Code asks which kernel to use: pick your Python. Run the setup cell first, then work down. Every query is wrapped in `q(""" ... """)`, so you only ever write SQL between the quotes.

When you reach a challenge, have a go in the empty cell underneath. The answers are at the bottom of the notebook.

In [1]:
from _setup import q

## What's in the warehouse?

`SHOW TABLES` lists everything you can query. The `dim_` tables describe things (stores, products, dates) and the `fact_` tables record what happened (sales, footfall, stock).

In [2]:
q("""
SHOW TABLES
""")

,name
0,dim_currency
1,dim_date
2,dim_period
3,dim_product
4,dim_promo
5,dim_store
6,dq_check_results
7,dq_table_profile
8,fact_digital_sales
9,fact_digital_targets


## Look before you query

`DESCRIBE` lists a table's columns and their types. `LIMIT 5` shows a few rows, which is the quickest way to see what the data actually looks like. I do both with any table I haven't used before.

In [3]:
q("""
DESCRIBE dim_store
""")

,column_name,column_type,null,key,default,extra
0,store_id,VARCHAR,YES,None,None,None
1,store_name,VARCHAR,YES,None,None,None
2,channel,VARCHAR,YES,None,None,None
3,store_type,VARCHAR,YES,None,None,None
4,square_footage,BIGINT,YES,None,None,None
5,market_code,VARCHAR,YES,None,None,None
6,market_name,VARCHAR,YES,None,None,None
7,city,VARCHAR,YES,None,None,None
8,region,VARCHAR,YES,None,None,None
9,latitude,DOUBLE,YES,None,None,None


In [4]:
q("""
SELECT *
FROM dim_store
LIMIT 5
""")

,store_id,store_name,channel,store_type,square_footage,market_code,market_name,city,region,latitude,longitude,currency,is_home_market
0,ST0001,Areta Leeds,Retail,High Street,2526,UK,United Kingdom,Leeds,North England,53.79,-1.57,GBP,True
1,ST0002,Areta Chester,Retail,High Street,1594,UK,United Kingdom,Chester,North England,53.19,-2.92,GBP,True
2,ST0003,Areta London,Retail,Retail Park,4678,UK,United Kingdom,London,London,51.52,-0.12,GBP,True
3,ST0004,Areta Liverpool,Retail,Shopping Centre,3245,UK,United Kingdom,Liverpool,North England,53.38,-2.97,GBP,True
4,ST0005,Areta Bristol,Retail,High Street,1884,UK,United Kingdom,Bristol,South West,51.47,-2.61,GBP,True


## Filtering and sorting

`WHERE` keeps the rows you want, and `ORDER BY` sorts them. Text goes in single quotes. `AND` means both conditions must be true.

In [5]:
q("""
SELECT store_name, store_type, city
FROM dim_store
WHERE market_name = 'Poland'
  AND channel = 'Retail'
ORDER BY store_name
""")

,store_name,store_type,city
0,Areta Gdansk,Retail Park,Gdansk
1,Areta Gdansk (2),High Street,Gdansk
2,Areta Katowice,Outlet,Katowice
3,Areta Lodz,Retail Park,Lodz
4,Areta Lodz (2),Retail Park,Lodz
5,Areta Poznan,Retail Park,Poznan
6,Areta Poznan (2),Retail Park,Poznan
7,Areta Poznan (3),High Street,Poznan
8,Areta Poznan (4),Retail Park,Poznan
9,Areta Poznan (5),Retail Park,Poznan


## Counting and grouping

`COUNT(*)` counts rows. `GROUP BY` splits the rows into groups first, so you get one count per group.

**The rule worth remembering:** every column in the `SELECT` that isn't inside `COUNT`, `SUM`, `AVG` and so on must also be in the `GROUP BY`.

In [6]:
q("""
SELECT market_name, channel, COUNT(*) AS stores
FROM dim_store
GROUP BY market_name, channel
ORDER BY market_name, channel
""")

,market_name,channel,stores
0,Czechia,Concession,8
1,Czechia,Online,1
2,Czechia,Retail,6
3,France,Concession,4
4,France,Online,1
...,...,...,...
28,Slovakia,Online,1
29,Slovakia,Retail,4
30,United Kingdom,Concession,63
31,United Kingdom,Online,1


## Adding up sales

`fact_sales` only has a date, not a business year, so this uses `v_sales_flat`: the sales table with the date, store and product columns already joined on. Joins are in the next notebook.

`ROUND(x, 0)` tidies the numbers, and `AS` names the result column.

In [7]:
q("""
SELECT major_product_group,
       ROUND(SUM(net_sales_gbp), 0) AS net_sales
FROM v_sales_flat
WHERE business_year = 2025
GROUP BY major_product_group
ORDER BY net_sales DESC
""")

,major_product_group,net_sales
0,Outerwear,"13,106,522"
1,Footwear,"7,861,056"
2,Midlayer,"7,052,133"
3,Legwear,"5,846,534"
4,Tops,"3,684,915"
5,Accessories,"1,311,986"
6,Camping & Equipment,"137,514"


## WHERE or HAVING?

`WHERE` filters rows *before* they're grouped. `HAVING` filters groups *after* they're added up. Here: product groups that sold more than £2M in BY25.

In [8]:
q("""
SELECT product_group,
       ROUND(SUM(net_sales_gbp), 0) AS net_sales
FROM v_sales_flat
WHERE business_year = 2025
GROUP BY product_group
HAVING SUM(net_sales_gbp) > 2000000
ORDER BY net_sales DESC
""")

,product_group,net_sales
0,Waterproof Shell,"3,466,242"
1,Waterproof Insulated Jacket,"2,629,066"
2,Fleece,"2,567,819"
3,Gilets & Bodywarmers,"2,471,037"
4,Boots,"2,397,073"
5,3 in 1,"2,096,125"


## Challenges

**1.1** How many stores are there of each `store_type`? Use `dim_store`.

In [9]:
# your query here
# q("""
#     SELECT ...
# """)

**1.2** What were the ten best-selling styles by units in BY25? Use `v_sales_flat` and `SUM(quantity)`.

*Hint: style names repeat across brands, so group by `style_name` and `brand_name` together.*

In [10]:
# your query here
# q("""
#     SELECT ...
# """)

**1.3** Net sales and gross margin % by brand for BY25.

*Hint: gross margin % is (net sales minus cost) divided by net sales, so `SUM(net_sales_gbp - cost_gbp) / SUM(net_sales_gbp)`. Never average a percentage: build it from its two totals.*

In [11]:
# your query here
# q("""
#     SELECT ...
# """)

**1.4** Which day of the week sells the most in BY25? `v_sales_flat` has a `day_name` column.

In [12]:
# your query here
# q("""
#     SELECT ...
# """)

---
## Answers

Have a go first. There's usually more than one right way to write these.

**1.1**

In [13]:
q("""
SELECT store_type, COUNT(*) AS stores
FROM dim_store
GROUP BY store_type
ORDER BY stores DESC
""")

,store_type,stores
0,Retail Park,98
1,High Street,76
2,Outlet,54
3,Shopping Centre,49
4,Garden Centre,42
5,Department Store,31
6,Online,11


**1.2**

In [14]:
q("""
SELECT style_name, brand_name, SUM(quantity) AS units
FROM v_sales_flat
WHERE business_year = 2025
GROUP BY style_name, brand_name
ORDER BY units DESC
LIMIT 10
""")

,style_name,brand_name,units
0,Chamonix Socks,Kestrel Ridge,"17,541"
1,Tatra Socks,Areta,"14,265"
2,Skiddaw Socks,Basecamp,"14,056"
3,Cuillin Gilets & Bodywarmers,Basecamp,"14,031"
4,Sarek Base Layer,Areta,"13,125"
5,Scafell Shoes,Basecamp,"12,454"
6,Eiger Fleece,Kestrel Ridge,"11,260"
7,Snowdon Fleece,Areta,"10,669"
8,Brecon Socks,Areta,"10,662"
9,Aconcagua Shorts,Areta,"10,638"


**1.3**

In [15]:
q("""
SELECT brand_name,
       ROUND(SUM(net_sales_gbp), 0) AS net_sales,
       ROUND(100 * SUM(net_sales_gbp - cost_gbp) / SUM(net_sales_gbp), 1) AS gross_margin_pct
FROM v_sales_flat
WHERE business_year = 2025
GROUP BY brand_name
ORDER BY net_sales DESC
""")

,brand_name,net_sales,gross_margin_pct
0,Areta,"13,515,529",68.70
1,Kestrel Ridge,"8,991,512",69.10
2,Basecamp,"8,455,692",67.30
3,Areta Pro,"8,037,927",70.30


**1.4** Saturday wins, which matches the trading calendar on the Sales page.

In [16]:
q("""
SELECT day_name, ROUND(SUM(net_sales_gbp), 0) AS net_sales
FROM v_sales_flat
WHERE business_year = 2025
GROUP BY day_name
ORDER BY net_sales DESC
""")

,day_name,net_sales
0,Saturday,"7,446,870"
1,Sunday,"6,201,345"
2,Friday,"5,851,181"
3,Monday,"4,908,488"
4,Tuesday,"4,896,031"
5,Wednesday,"4,881,568"
6,Thursday,"4,815,178"
